<a href="https://colab.research.google.com/github/akshay-aiml/LangChain_LangGraph/blob/main/RAG_Single_Query_Rewriting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install langchain-groq langgraph  langchain-community  faiss-cpu   tiktoken -qU langchain-text-splitters langchain-huggingface


In [13]:
from google.colab import userdata

api_key1 = userdata.get("GROQ_API_KEY")

In [4]:
from langchain_core.documents import Document
# Sample documents — our 'knowledge base'
docs = [
    Document(page_content="""
    Overfitting occurs when a machine learning model learns the training data too well,
    including its noise and outliers. The model performs excellently on training data
    but poorly on unseen test data. Common solutions include regularization (L1/L2),
    dropout, early stopping, and using more training data.
    """, metadata={"source": "ml_basics", "topic": "overfitting"}),

    Document(page_content="""
    Gradient Descent is an optimization algorithm used to minimize the loss function
    in machine learning. It works by iteratively moving in the direction of the negative
    gradient. Variants include Stochastic Gradient Descent (SGD), Mini-batch GD,
    Adam optimizer, and RMSprop. Learning rate is a critical hyperparameter.
    """, metadata={"source": "ml_basics", "topic": "optimization"}),

    Document(page_content="""
    Transformers are a deep learning architecture based on self-attention mechanisms.
    Introduced in the paper 'Attention is All You Need' (2017), they replaced RNNs
    for NLP tasks. Key components: multi-head attention, positional encoding,
    feed-forward layers, encoder-decoder structure. Used in BERT, GPT, T5.
    """, metadata={"source": "deep_learning", "topic": "transformers"}),

    Document(page_content="""
    RAG (Retrieval Augmented Generation) combines a retriever and a generator.
    It retrieves relevant documents from a knowledge base and passes them as context
    to an LLM to generate answers. This reduces hallucination and allows LLMs to
    access up-to-date or domain-specific information without retraining.
    """, metadata={"source": "llm_techniques", "topic": "rag"}),

    Document(page_content="""
    Vector databases store data as high-dimensional vectors (embeddings). They enable
    semantic similarity search using metrics like cosine similarity or dot product.
    Popular vector DBs: Pinecone, Weaviate, Chroma, FAISS, Qdrant. Essential for
    building RAG systems and semantic search applications.
    """, metadata={"source": "llm_techniques", "topic": "vector_db"}),

    Document(page_content="""
    Cross-validation is a technique to evaluate model generalization. K-Fold CV splits
    data into K subsets; the model trains on K-1 folds and tests on the remaining fold,
    repeated K times. Stratified K-Fold preserves class distribution. Helps detect
    overfitting and gives a more reliable performance estimate.
    """, metadata={"source": "ml_basics", "topic": "evaluation"}),

    Document(page_content="""
    BERT (Bidirectional Encoder Representations from Transformers) is a pre-trained
    language model by Google. It uses masked language modeling and next sentence
    prediction for pre-training. BERT is bidirectional — it reads text in both
    directions simultaneously. Used for classification, NER, QA tasks via fine-tuning.
    """, metadata={"source": "deep_learning", "topic": "bert"}),

    Document(page_content="""
    Embeddings are dense vector representations of data (text, images, etc.) in a
    continuous vector space. Semantically similar items are close together. Text
    embeddings from models like OpenAI text-embedding-ada-002 or sentence-transformers
    capture meaning. Used in semantic search, clustering, RAG, and classification.
    """, metadata={"source": "llm_techniques", "topic": "embeddings"}),
]

print(f"✅ Knowledge base created with {len(docs)} documents")

✅ Knowledge base created with 8 documents


In [7]:
from google.colab import userdata

api_key = userdata.get("HGF")

In [9]:
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding model
embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5", model_kwargs = {'trust_remote_code': True})

# Build FAISS vector store directly from documents
vectorstore = FAISS.from_documents(docs, embeddings)

# Create a base retriever (top-3 results)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ FAISS vector store built and retriever ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

✅ FAISS vector store built and retriever ready!


In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # or "mixtral-8x7b-32768", "gemma2-9b-it"
    api_key=api_key,       # or set GROQ_API_KEY env variable
    temperature=0
)

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key1,
    temperature=0
)

def format_docs(docs):
    """Format retrieved docs into a single context string."""
    return "\n\n".join([
        f"[Doc {i+1}] ({d.metadata.get('topic', 'N/A')}):\n{d.page_content.strip()}"
        for i, d in enumerate(docs)
    ])

# Final answer prompt
answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say 'I don't have enough information.'

Context:
{context}

Question: {question}

Answer:
""")

answer_chain = answer_prompt | llm | StrOutputParser()

print("✅ LLM and answer chain ready!")

✅ LLM and answer chain ready!


In [17]:
#  Single Query Rewriting
#  Take the user's original (possibly messy/vague) query and rewrite it into a cleaner, more retrieval-friendly version using an LLM.
##  User Query → [LLM Rewriter] → Better Query → Retriever → LLM Answer

In [15]:
# ─── Query Rewriting Prompt ───────────────────────────────────────────────────
rewrite_prompt = ChatPromptTemplate.from_template("""
You are an expert at reformulating questions to improve document retrieval.

Rewrite the following question to be:
- More specific and descriptive
- Uses proper technical terminology
- Clear about what information is being sought

Original question: {question}

Rewritten question (return ONLY the rewritten question, no explanation):
""")

# Chain: rewrite query
rewrite_chain = rewrite_prompt | llm | StrOutputParser()


def single_rewrite_rag(question: str, verbose: bool = True) -> str:
    """
    RAG pipeline with single query rewriting.

    Steps:
      1. Rewrite the original query using LLM
      2. Retrieve docs using the rewritten query
      3. Generate final answer
    """
    # Step 1: Rewrite
    rewritten_query = rewrite_chain.invoke({"question": question})

    if verbose:
        print(f"📝 Original Query  : {question}")
        print(f"✏️  Rewritten Query : {rewritten_query}")
        print("-" * 60)

    # Step 2: Retrieve using rewritten query
    retrieved_docs = base_retriever.invoke(rewritten_query)

    if verbose:
        print(f"📚 Retrieved {len(retrieved_docs)} docs:")
        for d in retrieved_docs:
            print(f"   → [{d.metadata.get('topic')}]")
        print("-" * 60)

    # Step 3: Answer
    context = format_docs(retrieved_docs)
    answer = answer_chain.invoke({"context": context, "question": question})

    return answer




In [16]:
# ─── Test It ──────────────────────────────────────────────────────────────────
result = single_rewrite_rag("why does my model not work well on new data?")
print("\n🤖 Answer:")
print(result)

📝 Original Query  : why does my model not work well on new data?
✏️  Rewritten Query : What are the primary causes of poor generalization performance in my machine learning model when applied to unseen, out-of-sample data, and what techniques can be employed to improve its predictive accuracy and robustness on new, unseen datasets?
------------------------------------------------------------
📚 Retrieved 3 docs:
   → [overfitting]
   → [optimization]
   → [evaluation]
------------------------------------------------------------

🤖 Answer:
Your model may not work well on new data because it is overfitting, meaning it has learned the training data too well, including its noise and outliers, and therefore performs poorly on unseen test data.
